In [63]:
from pykalshi import KalshiClient, Action, Side
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

client = KalshiClient()

In [30]:
from pykalshi import MarketStatus, History, AsyncHistory
from decimal import Decimal

markets = client.history.get_markets(series_ticker='KXNHLGAME')

market = min(markets, key=lambda m: Decimal(m.volume_fp or "0"))

market

Ticker,KXNHLGAME-26MAR12CGYNJ-CGY
Title,Calgary at New Jersey Winner?
Status,finalized
YES,$0.9900 / $1.0000
Last,$0.9900
Volume,76604.00
Open Int,0.00
Result,YES
Closes,"Mar 13, 02:15"


In [ ]:
from pykalshi import CandlestickPeriod
from datetime import datetime, timedelta

def iso_to_ts(s: str) -> int:
    return int(datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp())

end = iso_to_ts(market.close_time)
start = end - int(timedelta(days=0.2).total_seconds())

candles = market.get_candlesticks(start_ts=start, end_ts=end, period=CandlestickPeriod.ONE_MINUTE)


candles_df = candles.to_dataframe()

candles_df


,ticker,end_period_ts,volume_fp,open_interest_fp,open_dollars,high_dollars,low_dollars,close_dollars,mean_dollars,timestamp
0,KXNHLGAME-26MAR12CGYNJ-CGY,1773350880,0.00,2719.00,NaN,NaN,NaN,NaN,NaN,2026-03-12 21:28:00
1,KXNHLGAME-26MAR12CGYNJ-CGY,1773351060,25.00,2744.00,0.3700,0.3700,0.3700,0.3700,0.3700,2026-03-12 21:31:00
2,KXNHLGAME-26MAR12CGYNJ-CGY,1773351420,25.00,2769.00,0.3700,0.3700,0.3700,0.3700,0.3700,2026-03-12 21:37:00
3,KXNHLGAME-26MAR12CGYNJ-CGY,1773351540,1.00,2770.00,0.3700,0.3700,0.3700,0.3700,0.3700,2026-03-12 21:39:00
4,KXNHLGAME-26MAR12CGYNJ-CGY,1773351660,0.00,2770.00,NaN,NaN,NaN,NaN,NaN,2026-03-12 21:41:00
...,...,...,...,...,...,...,...,...,...,...
189,KXNHLGAME-26MAR12CGYNJ-CGY,1773366180,0.00,46706.00,NaN,NaN,NaN,NaN,NaN,2026-03-13 01:43:00
190,KXNHLGAME-26MAR12CGYNJ-CGY,1773366240,0.00,46706.00,NaN,NaN,NaN,NaN,NaN,2026-03-13 01:44:00
191,KXNHLGAME-26MAR12CGYNJ-CGY,1773366300,0.00,46706.00,NaN,NaN,NaN,NaN,NaN,2026-03-13 01:45:00
192,KXNHLGAME-26MAR12CGYNJ-CGY,1773366360,0.00,46706.00,NaN,NaN,NaN,NaN,NaN,2026-03-13 01:46:00


In [62]:
# @title
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = candles.to_dataframe()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.7, 0.3],
)

# Candlesticks
fig.add_trace(
    go.Candlestick(
        x=df["timestamp"],
        open=df["open_dollars"].astype(float),
        high=df["high_dollars"].astype(float),
        low=df["low_dollars"].astype(float),
        close=df["close_dollars"].astype(float),
        increasing_line_color="#22c55e",
        decreasing_line_color="#ef4444",
        name="Price",
    ),
    row=1, col=1,
)

# Volume bars — colored by direction
colors = [
    "#22c55e" if c >= o else "#ef4444"
    for c, o in zip(df["close_dollars"].astype(float), df["open_dollars"].astype(float))
]
fig.add_trace(
    go.Bar(
        x=df["timestamp"],
        y=df["volume_fp"].astype(float),
        marker_color=colors,
        opacity=0.5,
        name="Volume",
    ),
    row=2, col=1,
)

fig.update_layout(
    title=f"{market.ticker}  —  Daily",
    template="plotly_dark",
    height=500,
    showlegend=False,
    xaxis_rangeslider_visible=False,
    margin=dict(l=50, r=20, t=50, b=30),
    yaxis=dict(title="Price ($)", side="right"),
    yaxis2=dict(title="Volume", side="right"),
)

fig.show()

In [61]:
def clean_candles_df(candles_df: pd.DataFrame) -> pd.DataFrame:
    df = candles_df.copy()

    price_cols = [
        "open_dollars",
        "high_dollars",
        "low_dollars",
        "close_dollars",
        "mean_dollars",
    ]

    # Convert price columns to numeric
    for col in price_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows where close is missing
    # close is required for returns / realized volatility
    df = df.dropna(subset=["close_dollars"])

    # Fill missing OHLC/mean values using close
    for col in ["open_dollars", "high_dollars", "low_dollars", "mean_dollars"]:
        df[col] = df[col].fillna(df["close_dollars"])

    # Sort data
    df = df.sort_values(["ticker", "end_period_ts"]).reset_index(drop=True)

    return df    

cleaned_candles_df = clean_candles_df(candles_df)

In [72]:
def iso_to_ts(s: str) -> int:
    return int(datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp())

def add_surface_axes(
    candles_df: pd.DataFrame,
    close_time: str,
    price_col: str = "close_dollars",
    eps: float = 1e-6,
) -> pd.DataFrame:
    df = candles_df.copy()

    close_ts = iso_to_ts(close_time)

    # Convert price to numeric first
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    # Drop rows where price is missing
    df = df.dropna(subset=[price_col])

    # Price is probability-like value
    df["price"] = df[price_col].clip(lower=eps, upper=1 - eps)

    # Odds and log-odds
    df["odds"] = df["price"] / (1 - df["price"])
    df["log_odds"] = np.log(df["odds"])

    # Time until market close, in days
    df["time_to_close_minutes"] = (close_ts - df["end_period_ts"]) / (60)

    df = df[
        (df["time_to_close_minutes"] > 0)
        & np.isfinite(df["log_odds"])
    ].copy()

    return df

In [73]:
surface_df = add_surface_axes(candles_df, market.close_time)

surface_df[[
    "ticker",
    "end_period_ts",
    "price",
    "odds",
    "log_odds",
    "time_to_close_minutes",
]].head()

,ticker,end_period_ts,price,odds,log_odds,time_to_close_minutes
1,KXNHLGAME-26MAR12CGYNJ-CGY,1773351060,0.37,0.587302,-0.532217,284.066667
2,KXNHLGAME-26MAR12CGYNJ-CGY,1773351420,0.37,0.587302,-0.532217,278.066667
3,KXNHLGAME-26MAR12CGYNJ-CGY,1773351540,0.37,0.587302,-0.532217,276.066667
5,KXNHLGAME-26MAR12CGYNJ-CGY,1773351780,0.37,0.587302,-0.532217,272.066667
8,KXNHLGAME-26MAR12CGYNJ-CGY,1773352800,0.37,0.587302,-0.532217,255.066667


In [78]:
import numpy as np
import pandas as pd

def add_realized_volatility(
    surface_df: pd.DataFrame,
    ticker_col: str = "ticker",
    ts_col: str = "end_period_ts",
    value_col: str = "log_odds",
    rolling_window: int = 10,
) -> pd.DataFrame:
    df = surface_df.copy()

    # Sort before calculating diffs
    df = df.sort_values([ticker_col, ts_col]).reset_index(drop=True)

    # Return in log-odds space
    df["log_odds_return"] = df.groupby(ticker_col)[value_col].diff()

    # Rolling realized volatility by ticker
    df["realized_vol"] = (
        df.groupby(ticker_col)["log_odds_return"]
        .rolling(rolling_window)
        .std()
        .reset_index(level=0, drop=True)
    )

    # Estimate candles per day from timestamps
    median_step_seconds = df.groupby(ticker_col)[ts_col].diff().median()

    if pd.isna(median_step_seconds) or median_step_seconds <= 0:
        periods_per_day = 24
    else:
        periods_per_day = int((24 * 60 * 60) / median_step_seconds)

    # Scale to daily realized volatility
    df["realized_vol_daily"] = df["realized_vol"] * np.sqrt(periods_per_day)

    df = df.dropna(subset=["log_odds_return", "realized_vol_daily"])

    return df

In [79]:
vol_df = add_realized_volatility(
    surface_df,
    rolling_window=10,
)

vol_df[[
    "ticker",
    "end_period_ts",
    "price",
    "log_odds",
    "time_to_close_minutes",
    "log_odds_return",
    "realized_vol_daily",
]].head()

,ticker,end_period_ts,price,log_odds,time_to_close_minutes,log_odds_return,realized_vol_daily
10,KXNHLGAME-26MAR12CGYNJ-CGY,1773355140,0.37,-0.532217,216.066667,0.0,0.0
11,KXNHLGAME-26MAR12CGYNJ-CGY,1773355440,0.37,-0.532217,211.066667,0.0,0.0
12,KXNHLGAME-26MAR12CGYNJ-CGY,1773355680,0.37,-0.532217,207.066667,0.0,0.0
13,KXNHLGAME-26MAR12CGYNJ-CGY,1773355800,0.37,-0.532217,205.066667,0.0,0.0
14,KXNHLGAME-26MAR12CGYNJ-CGY,1773355980,0.37,-0.532217,202.066667,0.0,0.0


In [80]:
import numpy as np
import pandas as pd

def add_realized_volatility(
    surface_df: pd.DataFrame,
    ticker_col: str = "ticker",
    ts_col: str = "end_period_ts",
    value_col: str = "log_odds",
    rolling_window: int = 10,
) -> pd.DataFrame:
    df = surface_df.copy()

    df[ts_col] = pd.to_numeric(df[ts_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    df = df.dropna(subset=[ticker_col, ts_col, value_col])
    df = df.sort_values([ticker_col, ts_col]).reset_index(drop=True)

    # Change in log-odds from previous candle
    df["log_odds_return"] = df.groupby(ticker_col)[value_col].diff()

    # Rolling realized volatility
    df["realized_vol"] = (
        df.groupby(ticker_col)["log_odds_return"]
        .rolling(rolling_window)
        .std()
        .reset_index(level=0, drop=True)
    )

    df = df.dropna(subset=["realized_vol"]).copy()

    return df

In [81]:
vol_df = add_realized_volatility(
    surface_df,
    rolling_window=10,
)

vol_df[[
    "ticker",
    "end_period_ts",
    "price",
    "log_odds",
    "time_to_close_minutes",
    "log_odds_return",
    "realized_vol",
]].head()

,ticker,end_period_ts,price,log_odds,time_to_close_minutes,log_odds_return,realized_vol
10,KXNHLGAME-26MAR12CGYNJ-CGY,1773355140,0.37,-0.532217,216.066667,0.0,0.0
11,KXNHLGAME-26MAR12CGYNJ-CGY,1773355440,0.37,-0.532217,211.066667,0.0,0.0
12,KXNHLGAME-26MAR12CGYNJ-CGY,1773355680,0.37,-0.532217,207.066667,0.0,0.0
13,KXNHLGAME-26MAR12CGYNJ-CGY,1773355800,0.37,-0.532217,205.066667,0.0,0.0
14,KXNHLGAME-26MAR12CGYNJ-CGY,1773355980,0.37,-0.532217,202.066667,0.0,0.0


In [82]:
import plotly.graph_objects as go

def plot_realized_vol_points(vol_df):
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=vol_df["log_odds"],
                y=vol_df["time_to_close_minutes"],
                z=vol_df["realized_vol"],
                mode="markers",
                marker=dict(
                    size=3,
                    opacity=0.75,
                    color=vol_df["realized_vol"],
                    colorbar=dict(title="Realized Vol"),
                ),
                text=[
                    f"Ticker: {ticker}<br>"
                    f"Price: {price:.4f}<br>"
                    f"Log Odds: {log_odds:.4f}<br>"
                    f"Time to Close: {t:.1f} min<br>"
                    f"Realized Vol: {rv:.6f}"
                    for ticker, price, log_odds, t, rv in zip(
                        vol_df["ticker"],
                        vol_df["price"],
                        vol_df["log_odds"],
                        vol_df["time_to_close_minutes"],
                        vol_df["realized_vol"],
                    )
                ],
                hoverinfo="text",
            )
        ]
    )

    fig.update_layout(
        title="Realized Volatility Points",
        scene=dict(
            xaxis_title="Log Odds",
            yaxis_title="Time Till Close (minutes)",
            zaxis_title="Realized Volatility",
        ),
        width=1000,
        height=750,
    )

    fig.show()

In [83]:
plot_realized_vol_points(vol_df)

In [84]:
import numpy as np
from scipy.interpolate import griddata
import plotly.graph_objects as go

def plot_realized_vol_surface(vol_df, grid_size=60):
    x = vol_df["log_odds"].values
    y = vol_df["time_to_close_minutes"].values
    z = vol_df["realized_vol"].values

    x_grid = np.linspace(x.min(), x.max(), grid_size)
    y_grid = np.linspace(y.min(), y.max(), grid_size)

    X, Y = np.meshgrid(x_grid, y_grid)

    Z = griddata(
        points=(x, y),
        values=z,
        xi=(X, Y),
        method="linear",
    )

    Z_nearest = griddata(
        points=(x, y),
        values=z,
        xi=(X, Y),
        method="nearest",
    )

    Z = np.where(np.isnan(Z), Z_nearest, Z)

    fig = go.Figure(
        data=[
            go.Surface(
                x=X,
                y=Y,
                z=Z,
            )
        ]
    )

    fig.update_layout(
        title="Realized Volatility Surface",
        scene=dict(
            xaxis_title="Log Odds",
            yaxis_title="Time Till Close (minutes)",
            zaxis_title="Realized Volatility",
        ),
        width=1000,
        height=750,
    )

    fig.show()

In [85]:
plot_realized_vol_surface(vol_df)